# Black-Scholes Model

Il modello di Black-Scholes è un framework matematico per la prezzatura dei contratti derivati, in particolare le opzioni di tipo europeo. Sviluppato nel 1973, ha rivoluzionato i mercati finanziari fornendo un metodo standardizzato per calcolare il valore teorico (Fair Value) di un'opzione, isolando il rischio.

## 1. Assunti Fondamentali (Le Ipotesi del Modello)

Per far funzionare la matematica, il modello vive in un mercato "ideale" e fa alcune assunzioni forti (che nella realtà spesso non si verificano):

* **Opzioni Europee:** L'opzione può essere esercitata *solo* alla data di scadenza (non prima, come nelle opzioni Americane).
* **Distribuzione Log-Normale:** I rendimenti del sottostante seguono una distribuzione normale (i prezzi non possono essere negativi e seguono un moto browniano geometrico).
* **Volatilità Costante:** La volatilità del sottostante è nota e costante per tutta la vita dell'opzione (questo è il limite più grande del modello).
* **Assenza di Dividendi:** Il titolo sottostante non paga dividendi durante la vita dell'opzione (esistono estensioni del modello per includerli).
* **Mercato Frizionale:** Non ci sono costi di transazione, tasse o limiti alle vendite allo scoperto. Il tasso privo di rischio ($r$) è costante.

## 2. Le 5 Variabili di Input

Per prezzare un'opzione con Black-Scholes, il modello ha bisogno di esattamente 5 ingredienti:

1. $S$: Prezzo *Spot* (attuale) del titolo sottostante.
2. $K$: Prezzo di *Strike* (il prezzo a cui si ha diritto di comprare/vendere).
3. $T$: Tempo a scadenza (espresso in frazione di anno, es. 6 mesi = 0.5).
4. $r$: Tasso di interesse *Risk-Free* (es. rendimento dei titoli di stato).
5. $\sigma$: Volatilità implicita del titolo sottostante (deviazione standard annualizzata dei rendimenti).

## 3. Le Formule Matematiche

Il prezzo teorico di un'opzione Call ($C$) e di un'opzione Put ($P$) si calcola con le seguenti equazioni differenziali stocastiche:

### Opzione Call (Diritto di Acquistare)
$$C=S \cdot N(d_1) - K \cdot e^{-rT} \cdot N(d_2)$$

### Opzione Put (Diritto di Vendere)
$$P=K \cdot e^{-rT} \cdot N(-d_2) - S \cdot N(-d_1)$$

**Dove i parametri di probabilità $d_1$ e $d_2$ sono calcolati come:**

$$d_1=\frac{\ln(S/K) + (r + \frac{\sigma^2}{2})T}{\sigma \sqrt{T}}$$

$$d_2=d_1 - \sigma \sqrt{T}$$

*Nota:* $N(x)$ rappresenta la funzione di ripartizione della distribuzione normale standardizzata (indica la probabilità che l'opzione scada "In The Money").

In [5]:
import numpy as np
from scipy.stats import norm

def black_scholes(S, K, T, r, sigma, q=0, option_type='call'):
    """
    Black-Scholes option pricing model
    
    Parameters:
    S: Current stock price
    K: Strike price
    T: Time to expiration (in years)
    r: Risk-free interest rate
    sigma: Volatility (annualized standard deviation)
    q: Dividend yield (default 0)
    option_type: 'call' or 'put'
    
    Returns:
    Option price
    """
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == 'call':
        price = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:  # put
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
    
    return price

# Example usage
S = 100      # Stock price
K = 100      # Strike price
T = 1        # 1 year to expiration
r = 0.05     # 5% risk-free rate
sigma = 0.2  # 20% volatility
q = 0        # 0% dividend yield

call_price = black_scholes(S, K, T, r, sigma, q=q, option_type='call')
put_price = black_scholes(S, K, T, r, sigma, q=q, option_type='put')

print(f"Call option price: ${call_price:.2f}")
print(f"Put option price: ${put_price:.2f}")

Call option price: $10.45
Put option price: $5.57


## 4. Oltre il Prezzo: Le "Greche"

Il vero potere di Black-Scholes non è solo trovare il prezzo, ma capire come questo prezzo cambia se cambia uno degli input. Queste derivate parziali sono chiamate "Greche":

* **Delta ($\Delta$):** Quanto cambia il prezzo dell'opzione se il sottostante ($S$) si muove di 1$.
* **Gamma ($\Gamma$):** L'accelerazione del Delta (la derivata seconda rispetto a $S$).
* **Theta ($\Theta$):** Quanto valore perde l'opzione ogni giorno che passa (Decadimento Temporale, derivata rispetto a $T$).
* **Vega ($\nu$):** Quanto cambia il prezzo se la volatilità ($\sigma$) aumenta dell'1%.
* **Rho ($\rho$):** La sensibilità ai tassi di interesse ($r$).

## 5. Limiti Pratici per il Trading Reale

Nel mondo reale, i mercati subiscono "Cigni Neri" (code grasse nella distribuzione) e la volatilità non è mai costante, ma cambia in base allo strike (fenomeno del *Volatility Smile*). Questo modello è una bussola teorica eccellente, ma i prezzi a mercato differiranno sempre leggermente da $C$ e $P$.

In [ ]:
# Calcola le "Greche" (Delta, Gamma, Theta, Vega, Rho) per call e put usando le variabili già presenti:
# S, K, T, r, sigma, q e le funzioni/moduli importati (np, norm).

def _d1_d2(S, K, T, r, sigma, q):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2

def greeks(S, K, T, r, sigma, q=0):
    d1, d2 = _d1_d2(S, K, T, r, sigma, q)
    pdf_d1 = norm.pdf(d1)
    cdf_d1 = norm.cdf(d1)
    cdf_d2 = norm.cdf(d2)
    exp_qT = np.exp(-q * T)
    exp_rT = np.exp(-r * T)

    # Delta
    delta_call = exp_qT * cdf_d1
    delta_put  = exp_qT * (cdf_d1 - 1.0)

    # Gamma (stesso per call e put)
    gamma = exp_qT * pdf_d1 / (S * sigma * np.sqrt(T))

    # Vega (sensibilità per 1 punto di volatilità, es. da 0.20 a 0.21)
    vega = S * exp_qT * pdf_d1 * np.sqrt(T)

    # Theta (espresso in unità annue). Fornisco anche il valore giornaliero dividendo per 365.
    theta_call = (
        - (S * pdf_d1 * sigma * exp_qT) / (2 * np.sqrt(T))
        - r * K * exp_rT * cdf_d2
        + q * S * exp_qT * cdf_d1
    )
    theta_put = (
        - (S * pdf_d1 * sigma * exp_qT) / (2 * np.sqrt(T))
        + r * K * exp_rT * norm.cdf(-d2)
        - q * S * exp_qT * norm.cdf(-d1)
    )

    # Rho (sensibilità ai tassi, per 1 punto di tasso, es. da 0.05

r"""# La Volatilità Implicita (IV): Il "Battito Cardiaco" del Mercato

Mentre la *Volatilità Storica* misura quanto un titolo si è mosso nel passato, la **Volatilità Implicita (Implied Volatility - IV)** è una misura prospettica (forward-looking). Rappresenta l'aspettativa del mercato su quanto il prezzo del sottostante oscillerà in futuro, da oggi fino alla scadenza dell'opzione.

In termini pratici, l'IV è il "prezzo della paura o dell'avidità" sul mercato.

## 1. Il Concetto Matematico (Reverse Engineering)

Nel modello di Black-Scholes classico, inseriamo 5 variabili (inclusa la volatilità storica $\sigma$) per calcolare il prezzo teorico ($C$ o $P$). 
Tuttavia, nel mondo reale, il prezzo dell'opzione ($C_{mercato}$) è già deciso dalla domanda e dall'offerta sul mercato. 

Quello che facciamo con la Volatilità Implicita è usare il modello "al contrario". Inseriamo il prezzo reale di mercato e troviamo qual è l'unico valore di volatilità ($\sigma_{imp}$) che giustifica quel prezzo:

$$C_{mercato}=BS(S, K, T, r, \sigma_{imp})$$

*Nota matematica:* Non esiste una formula algebrica per isolare $\sigma_{imp}$ dall'equazione di Black-Scholes. Per trovarla, i computer utilizzano metodi numerici di approssimazione (ricerca per tentativi), come il **Metodo di Newton-Raphson**.

## 2. Come si interpreta l'IV nel Trading?

Per un fondo d'investimento, l'IV è la bussola per capire se un'opzione è "cara" o "economica":

* **IV Alta:** Le opzioni costano molto. Il mercato si aspetta un forte movimento (es. prima della pubblicazione degli utili trimestrali o di un dato sull'inflazione). Spesso, i fondi istituzionali *vendono* opzioni quando l'IV è alta per incassare premi ricchi.
* **IV Bassa:** Le opzioni sono economiche. Il mercato è calmo. Questo è il momento in cui i fondi *comprano* opzioni per proteggere il portafoglio a basso costo.

## 3. L'IV Rank e l'IV Percentile

Dire "L'azione Apple ha un'IV del 30%" significa poco senza contesto. Il 30% è alto o basso per Apple? Per capirlo si usano due metriche relative:

* **IV Rank:** Confronta l'IV attuale con il suo massimo e minimo delle ultime 52 settimane. (Un Rank di 100 significa che la volatilità è al massimo annuale).
* **IV Percentile:** Indica la percentuale di giorni, nell'ultimo anno, in cui la volatilità è stata inferiore a quella di oggi.

## 4. Il "Volatility Smile" (Il limite di Black-Scholes)

Il modello di Black-Scholes assume che la volatilità sia costante per tutti i prezzi di strike ($K$). 
Tuttavia, dopo il crollo del mercato del 1987, i trader hanno iniziato a prezzare un rischio maggiore per crolli estremi. 

Se tracciamo l'IV di tutte le opzioni con la stessa scadenza ma strike diversi, non otteniamo una linea piatta, ma una curva a forma di "sorriso" (Smile) o di "ghigno" (Skew). Le opzioni Put molto lontane dal prezzo attuale (Out-Of-The-Money) hanno un'IV molto più alta delle opzioni Call, perché gli investitori pagano un sovrapprezzo per assicurarsi contro i crolli di mercato ("Tail Risk").

In [17]:
def implied_vol_newton(market_price, S, K, T, r, q=0.0, option_type='call',
                       initial_sigma=0.2, tol=1e-8, max_iter=100):
    """
    Calcola la volatilità implicita tramite il metodo di Newton-Raphson.
    Dipende dalla funzione `black_scholes` e da `_d1_d2` già presenti nel notebook.
    """
    sigma = float(initial_sigma)
    for i in range(max_iter):
        price = black_scholes(S, K, T, r, sigma, q=q, option_type=option_type)
        diff = price - market_price
        if abs(diff) < tol:
            return sigma

        # calcola d1 per la vega usando la funzione _d1_d2 già definita
        d1, _ = _d1_d2(S, K, T, r, sigma, q)
        vega = S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(max(T, 1e-12))

        # se vega è troppo piccolo, Newton può fallire: interrompi
        if vega < 1e-12:
            break

        # aggiornamento Newton-Raphson
        sigma -= diff / vega

        # evitare valori non fisici
        if sigma <= 0 or np.isnan(sigma):
            sigma = 1e-8

    raise RuntimeError(f"Implied vol did not converge after {max_iter} iterations (last diff={diff})")


# Esempio di utilizzo con le variabili già presenti nel notebook:
implied_vol_call = implied_vol_newton(call_price, S, K, T, r, q=q, option_type='call', initial_sigma=sigma)
implied_vol_put  = implied_vol_newton(put_price,  S, K, T, r, q=q, option_type='put',  initial_sigma=sigma)

print(f"Implied vol (call) = {implied_vol_call:.6f}")
print(f"Implied vol (put)  = {implied_vol_put:.6f}")

Implied vol (call) = 0.200000
Implied vol (put)  = 0.200000
